# Day 5 Lab: Delta Live Tables (DLT) - Medallion Pipeline

## Objectives
- Build Bronze, Silver, Gold layers using DLT
- Apply data quality checks
- Understand declarative pipelines


## Step 1: Setup

Note: This works only within Pipelines and Jobs platform. DON'T TRY TO DIRECTLY ON THE NOTEBOOK.

Upload orders.csv to DBFS: dbfs:/Volumes/[catalog_name]/[schema_name]/[volume_name]/orders.csv

In [0]:
from pyspark import pipelines as dp
from pyspark.sql.functions import *

In [0]:
@dp.table(name='bronze_orders')
def bronze_orders():
    # update with proper path
    # return spark.read.format('csv').option('header', True).load('/Volumes/<catalog_name>/<schema_name>/<volume_name>/orders.csv')
    return spark.read.format('csv').option('header', True).load('/Volumes/dbcuser01/ingesting_data/myfiles/orders.csv')

In [0]:
@dp.table(name='silver_orders')
@dp.expect('valid_amount', 'amount IS NOT NULL AND amount > 0')
def silver_orders():
    df = dlt.read('bronze_orders')
    return df.filter(col('status') == 'completed').withColumn('amount', col('amount').cast('int'))

In [0]:
@dp.table(name='gold_customer_spend')
def gold_customer_spend():
    df = dlt.read('silver_orders')
    return df.groupBy('customer_id').agg(sum('amount').alias('total_spent'))

In [0]:
%sql
SELECT * FROM gold_customer_spend ORDER BY total_spent DESC